# 카페 데이터 전처리 노트북


1부: JSON → 전처리 CSV
2부: 전처리 CSV → Kiwi 명사·불용어 → PKL

첫 코드 셀에서 경로를 설정한 뒤 순서대로 실행하세요.


## 제출용 읽기 안내

**분석 목적**  
카페 크롤 결과를 표 형태로 정리하고, 중복·결측·텍스트 정제 후 명사 추출까지 진행한다.

**입력 데이터**  
`data/cafe_only/의대증원_카페_v2.json` 및 카페 전처리 CSV

**주요 산출물**  
카페 단독 전처리 CSV/PKL, 통합 단계에서 사용할 명사 컬럼

**코드 해석 시 유의점**  
본문 유사도 검사는 제목이 같은 글끼리 비교해 중복 제거 범위를 줄인다.

**해석 작성 칸**  
- 실행 결과에서 확인한 핵심 변화:
- 결과가 연구 질문에 주는 의미:
- 추가로 확인할 점:


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths
PROJECT_ROOT = setup_paths()


# 1부 — JSON → CSV


### 파일 불러오기


In [ ]:
# 파일 불러오기

import pandas as pd
import re
import json

# 1. JSON 파일 로드
# 각 플랫폼별로 크롤링된 데이터를 데이터프레임으로 불러옵니다.
df_cafe = pd.read_json(DATA_CAFE_ONLY / '의대증원_카페_v2.json')

print(f"네이버 카페 데이터 개수: {len(df_cafe)}")

df_cafe.head(50)


### 본문, 댓글 비어있는 행 제거


In [ ]:
# 본문, 댓글 비어있는 행 제거
import pandas as pd

# apply 함수를 사용하여 각 행의 리스트 길이를 체크
zero_comment_list_cnt = df_cafe[df_cafe['comment_list'].apply(len) == 0].shape[0]
print(f"댓글 리스트가 비어있는 데이터 개수: {zero_comment_list_cnt}개")

empty_docs = df_cafe[df_cafe['doc'].isnull() | (df_cafe['doc'] == '')]

# 2. 개수 확인
print(f"본문이 비어있는 데이터 개수: {len(empty_docs)}개")


# 1. 본문도 있고 댓글 리스트도 있는 데이터만 필터링
# 조건 1: doc가 null이 아님 (notnull)
# 조건 2: doc가 빈 문자열이 아님 (!= '')
# 조건 3: comment_list의 길이가 0보다 큼 (apply len > 0)
df_filtered = df_cafe[
    (df_cafe['doc'].notnull()) & 
    (df_cafe['doc'].str.strip() != '') & 
    (df_cafe['comment_list'].apply(len) > 0) 
].copy()

# 2. 인덱스 재정렬
df_filtered.reset_index(drop=True, inplace=True)

# 3. 결과 확인
print(f"✅ 최종 필터링된 데이터 개수: {len(df_filtered)}개")
print(f"🗑️ 제거된 데이터(본문누락 or 댓글없음): {len(df_cafe) - len(df_filtered)}개")


### 인덱스 재정렬


In [ ]:
# 인덱스 재정렬
# 제목: title
# 본문: doc
# 페이지 좋아요 수: like
# 댓글 수: comment_cnt
# 댓글 리스트: comment_list
# comment_list = [{'comment_content': 댓글 내용,
#                  'comment_like': 댓글 공감 수,
#                   'comment_date': 댓글 작성일}]
# 채널 종류(blog/cafe): ch
# 작성일: date(형식: 2025-01-01)
# 구간: section (1-4구간)

import pandas as pd
import ast

# 원본 데이터 복사 (원본 보존용)
df_reindex = df_filtered.copy()

# ---------------------------------------------------------
# 1. 작성일(date) 포맷 변경 및 컬럼 생성
# 기존 'time' 컬럼의 "2025.06.29. 10:11" -> 앞 10자리 슬라이싱 후 '.'을 '-'로 치환
# ---------------------------------------------------------
df_reindex['date'] = df_reindex['time'].astype(str).str.slice(0, 10).str.replace('.', '-')

# ---------------------------------------------------------
# 2. 구간(section) 할당 로직
# ---------------------------------------------------------
# 정확한 비교를 위해 date를 datetime 타입으로 임시 변환
dt_series = pd.to_datetime(df_reindex['date'], errors='coerce')

def get_section(date_val):
    if pd.isna(date_val): return None
    if pd.Timestamp('2024-01-01') <= date_val <= pd.Timestamp('2024-03-31'): return 1
    elif pd.Timestamp('2024-04-01') <= date_val <= pd.Timestamp('2024-06-30'): return 2
    elif pd.Timestamp('2024-07-01') <= date_val <= pd.Timestamp('2024-12-31'): return 3
    elif pd.Timestamp('2025-01-01') <= date_val <= pd.Timestamp('2025-06-30'): return 4
    else: return 0 # 예외 구간 (설정하신 기간 외의 데이터)

df_reindex['section'] = dt_series.apply(get_section)

# ---------------------------------------------------------
# 3. comment_list 딕셔너리 키 및 날짜 포맷 수정
# ---------------------------------------------------------
def process_comments(comments_data):
    # 1. 아예 비어있는 리스트인 경우 안전하게 빈 리스트 반환
    if isinstance(comments_data, list):
        if len(comments_data) == 0:
            return []
        comments = comments_data
        
    # 2. 결측치(NaN)인 경우 (float 타입이면서 NaN일 때만 걸러냄)
    elif isinstance(comments_data, float) and pd.isna(comments_data):
        return []
        
    # 3. 문자열인 경우 (CSV에서 불러와서 '[]' 처럼 텍스트로 굳어있는 경우)
    elif isinstance(comments_data, str):
        # 텍스트 자체가 비어있거나 대괄호만 있는 경우
        if comments_data.strip() in ["", "[]", "NaN", "nan"]:
            return []
        try:
            # 문자열을 실제 리스트/딕셔너리 객체로 변환
            comments = ast.literal_eval(comments_data)
        except (ValueError, SyntaxError):
            return []
            
    # 4. 그 외의 예외적인 형태가 들어오면 빈 리스트 반환
    else:
        return []
        
    # --- 여기서부터는 정상적인 리스트 형태의 데이터 정제 ---
    new_comments = []
    for c in comments:
        # 가끔 딕셔너리가 아닌 이상한 값이 섞여 있을 때를 대비한 방어 코드
        if not isinstance(c, dict):
            continue
            
        raw_date = str(c.get('comment_date', ''))
        formatted_date = raw_date[:10].replace('.', '-') if raw_date else ''
        
        new_dict = {
            'comment_content': c.get('content', ''),
            'comment_like': c.get('like', 0),
            'comment_date': formatted_date
        }
        new_comments.append(new_dict)
        
    return new_comments

df_reindex['comment_list'] = df_reindex['comment_list'].apply(process_comments)

# ---------------------------------------------------------
# 4. 컬럼 삭제 및 이름 변경 (인덱스 정리)
# ---------------------------------------------------------
# 삭제 대상: ch(기존), img, div, query_from, query_to, time
cols_to_drop = ['ch', 'img', 'div', 'query_from', 'query_to', 'time']
df_reindex = df_reindex.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

# ch2를 ch로 변경
df_reindex = df_reindex.rename(columns={'ch2': 'ch'})

# 최종적으로 원하는 컬럼 순서로 정렬 (선택 사항)
final_cols = ['title', 'doc', 'like', 'comment_cnt', 'comment_list', 'ch', 'date', 'section']
df_reindex = df_reindex[final_cols]

print("✅ 데이터 전처리 및 구조화 완료!")


In [ ]:
df_reindex.head(5)


### 텍스트 기본 정제


In [ ]:
# 텍스트 기본 정제
# 불필요한 노이즈 제거

# 순서의 중요성: 반드시 텍스트 정제(HTML, 공백 제거 등)를 먼저 진행한 후 유사도 80% 검사를 해야 합니다. 공백이나 태그가 섞여 있으면 똑같은 내용의 글도 컴퓨터는 다른 글로 인식하기 때문입니다.
# SequenceMatcher 최적화: 2만 개의 본문을 모든 글과 1:1로 비교하면 연산량이 너무 많아 주피터가 뻗을 수 있습니다. 그래서 **groupby('title')**을 사용하여 "우선 제목이 같은 글들끼리만" 본문 80% 유사도 검사를 하도록 범위를 좁혀 연산 속도를 대폭 높였습니다. (연산량을 줄이기 위한 범위 제한입니다.)
# 댓글 보존: ㅋㅋ, ㅎㅎ 같은 자음만으로 이루어진 무의미한 댓글은 clean_text를 거치면 빈 문자열("")이 됩니다. if cleaned_content: 조건을 통해 이러한 쓰레기 데이터를 리스트에서 자연스럽게 날려버렸습니다.

import re
from difflib import SequenceMatcher

# ---------------------------------------------------------
# 1. 텍스트 정제 함수 정의 (조건 1, 2, 4 통합)
# ---------------------------------------------------------
import re
import pandas as pd

def clean_text(text):
    if pd.isna(text): 
        return ""
    
    text = str(text)
    
    # 1. 특정 문장 제거 (네이버 카페 기본 양식)
    text = re.sub(r'게시판 안내를 확인해\s*주세요!?', '', text)
    text = re.sub(r'이해원연구소 코어패스 인터그레이트 4월 문항 공모!?', '', text)
    text = re.sub(r'실거래가 게시글에 관한 의견을 묻습니다!?', '', text)
    text = re.sub(r'수만휘는 수험생 대학생 선생님 학부모님이 함께 모여 서로 돕고 응원하는 대한민국 최대의 대입 커뮤니티입니다 선배들의 따뜻한 조언과 회원들의 존중 속에서 함께 만들어가는 공간입니다 이용 규정 보기!?','', text)
    
    # (선택) 추가로 자주 나오는 양식문구도 이곳에 넣을 수 있습니다.
    # text = re.sub(r'※\s*미취학글,\s*과외구인.*', '', text) 
    
    # 2. 영문 시작 ~ com으로 끝나는 도메인/URL 제거
    text = re.sub(r'[a-zA-Z][a-zA-Z0-9\.\-]*com\b', '', text)
    
    # 3. 일반적인 http/https URL 제거
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 4. HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    
    # 5. 자음/모음 반복 (ㅋㅋㅋ, ㅎㅎㅎ, ㅠㅠㅠ 등) 제거
    text = re.sub(r'[ㄱ-ㅎㅏ-ㅣ]+', '', text)
    
    # 6. 특수문자 및 이모지 제거 (한글, 영문, 숫자, 공백만 남김)
    # 주의: 여기서 점(.)이 날아가기 때문에 반드시 도메인 제거(2번)를 이보다 먼저 해야 합니다.
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    
    # 7. 연속된 공백, \n, \t 등을 하나의 공백으로 치환하고 양쪽 여백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# (선택) 댓글에도 .com 링크가 있을 수 있으니 댓글도 다시 정제
# df['comment_list'] = df['comment_list'].apply(clean_comments_list) 


# ---------------------------------------------------------
# 2. comment_list 내부 텍스트 정제 함수
# ---------------------------------------------------------
def clean_comments_list(comments):
    # 1. 만약 리스트가 아니라 문자열('[...]' 형태)로 굳어있다면 리스트로 변환
    if isinstance(comments, str):
        try:
            comments = ast.literal_eval(comments)
        except (ValueError, SyntaxError):
            return comments # 변환 실패 시 원본 유지
            
    # 리스트가 아니면 그대로 반환
    if not isinstance(comments, list):
        return comments
    
    cleaned_comments = []
    for c in comments:
        if not isinstance(c, dict):
            continue
            
        # 2. [핵심 수정] 키 이름 방어 로직
        # 'comment_content'를 먼저 찾고, 없으면 원본 키인 'content'를 찾음
        original_text = c.get('comment_content', c.get('content', ''))
        
        # 텍스트 정제 함수 적용
        cleaned_content = clean_text(original_text)
        
        # 정제 후 내용이 남아있는 경우만 보존
        if cleaned_content:
            new_c = c.copy()
            # 현재 딕셔너리가 갖고 있는 키 이름에 맞춰서 업데이트
            if 'comment_content' in new_c:
                new_c['comment_content'] = cleaned_content
            else:
                new_c['content'] = cleaned_content
                
            cleaned_comments.append(new_c)
            
    return cleaned_comments
# ---------------------------------------------------------
# [실행 파트] 정제 함수 적용
# ---------------------------------------------------------

# 정제 함수 적용
df_reindex['title'] = df_reindex['title'].apply(clean_text)
df_reindex['doc'] = df_reindex['doc'].apply(clean_text)
df_reindex['comment_list'] = df_reindex['comment_list'].apply(clean_comments_list)

# 2. 내용이 너무 짧아진(노이즈만 있던) 게시글 1차 필터링
df_reindex = df_reindex[df_reindex['doc'].str.len() >= 5].reset_index(drop=True)

print(f"✅ 1단계 정제 완료!")

# ---------------------------------------------------------
# 3. 고급 중복 제거: 동일 제목 + 본문 90% 이상 일치 (조건 3)
# ---------------------------------------------------------
print("🔍 2단계: 유사도 80% 이상 중복 게시글을 탐색합니다...")

def is_similar(str1, str2, threshold=0.8):
    # SequenceMatcher를 사용해 두 문자열의 유사도 비율(0.0 ~ 1.0) 계산
    return SequenceMatcher(None, str1, str2).ratio() >= threshold

indices_to_drop = set()

# '제목'이 완전히 같은 그룹끼리 묶어서 검사 (속도 최적화)
for title, group in df.groupby('title'):
    if len(group) > 1:
        # 그룹 내 인덱스 목록
        idxs = group.index.tolist()
        
        # 인덱스끼리 1:1 비교
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                idx1, idx2 = idxs[i], idxs[j]
                
                # 이미 삭제 예정인 인덱스는 건너뜀
                if idx2 in indices_to_drop:
                    continue
                
                doc1 = df_reindex.loc[idx1, 'doc']
                doc2 = df_reindex.loc[idx2, 'doc']
                
                # 본문 유사도가 80% 이상이면 나중 글(idx2)을 삭제 대상에 추가
                if is_similar(doc1, doc2, threshold=0.8):
                    indices_to_drop.add(idx2)

# 중복 인덱스 최종 삭제
before_cnt = len(df_reindex)
df_reindex = df_reindex.drop(index=list(indices_to_drop)).reset_index(drop=True)
after_cnt = len(df_reindex)

print(f"✅ 정제 완료!")
print(f"📊 최종 남은 유효 데이터: {after_cnt}개")


In [ ]:
df_reindex.tail(30)


In [ ]:
# 변환할 숫자형 컬럼 목록
num_cols = ['like', 'comment_cnt', 'section']

for col in num_cols:
    if col in df_reindex.columns:
        # pd.to_numeric을 통해 숫자로 변환 (변환 불가한 값은 NaN 처리 -> 0으로 채움 -> 정수형 변환)
        df_reindex[col] = pd.to_numeric(df_reindex[col], errors='coerce').fillna(0).astype(int)

print("숫자형 데이터 변환이 완료되었습니다.")


In [ ]:
# 최종 데이터 확인
print(f"✅ 최종 전처리 완료 데이터 개수: {len(df_reindex)}개")

# CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
save_path = str(DATA_CAFE_ONLY / '의대증원_cafedata_preprocess.csv')
df_reindex.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"\n데이터가 성공적으로 저장되었습니다: {save_path}")


# 2부 — CSV → 명사 PKL


# 형태소 분석


In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트: notebooks/ 하위 어느 깊이에서 실행해도 동작
_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
from project_paths import DATA_CAFE_ONLY, DATA_INTEGRATED, CONFIG_STOPWORDS

# 파일 불러오기
import pandas as pd
import re

df = pd.read_csv(DATA_CAFE_ONLY / '의대증원_cafedata_preprocess.csv') 

print(f"네이버 카페 데이터 개수: {len(df)}")

df.head(10)


In [ ]:
from kiwipiepy import Kiwi
from tqdm import tqdm


# 1. 인스턴스(일꾼) 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

title_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
title_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(df))) :
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(df['title'][i])

    # 형태소 분석 ( Kiwi는 tokenize를 사룔)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장
    pos_result = [(t.form, t.tag) for t in tokens]
    title_token_list.append(pos_result)

    # 명사 추출
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.tag) > 1]
    title_token_noun.append(nouns)


In [ ]:
print(title_token_list)


In [ ]:
# 본문 명사 추출

# 1. 인스턴스(일꾼) 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

document_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
document_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(df))) :
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(df['doc'][i])

    # 형태소 분석 ( Kiwi는 tokenize를 사룔)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장
    pos_result = [(t.form, t.tag) for t in tokens]
    document_token_list.append(pos_result)

    # 명사 추출
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.tag) > 1]
    document_token_noun.append(nouns) 


In [ ]:
print(document_token_noun[:10])


In [ ]:
import ast
from tqdm import tqdm

comment_token_list = [] # 형태소 전체 결과 (단어, 태그) 리스트
comment_token_noun = [] # 2글자 이상 명사만 담는 리스트

for i in tqdm(range(len(df))):
    raw_comments = df['comment_list'][i]
    
    # 해당 게시글의 모든 댓글 결과를 합칠 임시 리스트
    post_full_tokens = []
    post_nouns_only_noun = []
    post_nouns_only_verb = []
    post_nouns_only_adj = []
    
    # 1. 데이터 타입 방어 (문자열일 경우 리스트로 변환)
    if isinstance(raw_comments, str):
        try:
            comments = ast.literal_eval(raw_comments)
        except:
            comments = []
    else:
        comments = raw_comments
        
    # 2. 분석 시작
    if isinstance(comments, list):
        for c in comments:
            if isinstance(c, dict):
                # 키 이름 확인 (comment_content)
                text = str(c.get('comment_content', ''))
                
                if text.strip():
                    tokens = kiwi.tokenize(text)
                    
                    for t in tokens:
                        # (1) 전체 결과 저장 (단어, 태그) 튜플 형태
                        post_full_tokens.append((t.form, t.tag))
                        
                        # (2) 명사 필터링 (2글자 이상 NNG, NNP)
                        if t.tag in ['NNG', 'NNP'] and len(t.form) > 1:
                            post_nouns_only_noun.append(t.form)
    
    # 게시글 단위로 최종 리스트에 추가
    comment_token_list.append(post_full_tokens)
    comment_token_noun.append(post_nouns_only_noun)


# 불용어 처리(stopwords-ko.txt 기반)


In [ ]:
import os
current_directory = os.getcwd()
print("현재 디렉토리:", current_directory)


In [ ]:
# 불용어 사전 기반 불용어 리스트 정리
f = open(CONFIG_STOPWORDS / 'stopwords-ko.txt', 'r', encoding="UTF-8")
st = f.readlines()
f.close()

stw = []
for i in range(0, len(st)):
    stw.append(st[i].rstrip('\n'))

stw


In [ ]:
# 각 문서의 제목과 본문 또는 댓글등에서 불용어 제거

for word in stw:
    for i in range(0, len(title_token_noun)) :
        # 리스트에 불용어가 있을 경우 제거
        while word in title_token_noun[i] :
            title_token_noun[i].remove(word)
    for j in range(0, len(document_token_noun)) : 
        while word in document_token_noun[j] :
            document_token_noun[j].remove(word)
    for l in range(0, len(comment_token_noun)) : 
        while word in comment_token_noun[l] :
            comment_token_noun[l].remove(word)


In [ ]:
# 문서파일 docs로 적용하여 각각의 불용어 제거
df['title_token_list_pos'] = title_token_list # 형태소와 품사 리스트
df['title_token_noun'] = title_token_noun # 명사 리스트

df['document_token_list_pos'] = document_token_list
df['document_token_noun'] = document_token_noun

df['comment_token_list_pos'] = comment_token_list
df['comment_token_noun'] = comment_token_noun


In [ ]:
# 불용어 처리 후 최종 파일 저장 및 불러오기
import pickle

f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press.pkl", 'wb')
pickle.dump(df, f)
f.close()


In [ ]:
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press.pkl", 'rb')
data = pickle.load(f)
f.close()
data = data.drop(columns=['comment_nouns'])
data


In [ ]:
data_drop_list = data.drop(columns=['title_token_list_pos', 'document_token_list_pos', 'comment_token_list_pos'])
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl", 'wb')
pickle.dump(data_drop_list, f)
f.close()


In [ ]:
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl", 'rb')
d = pickle.load(f)
f.close()
d
